# Classify all beach cells with best ConvNeXt Tiny

This notebook reads all complete cell matrices from `Data/Raw/beach_all`, loads the best saved context-aware ConvNeXt Tiny model, and writes per-date classification CSV files to `Results/bs_all_cells_classification`.

It does not copy any input `.txt` files from the external drive.

In [ ]:
from pathlib import Path
import sys

import torch

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (
    (PROJECT_ROOT / "Source").is_dir()
    and (PROJECT_ROOT / "Notebooks").is_dir()
):
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (
    (PROJECT_ROOT / "Source").is_dir()
    and (PROJECT_ROOT / "Notebooks").is_dir()
):
    raise RuntimeError(
        "Could not locate project root containing Source and Notebooks."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Source.inference import (
    classify_date_directories,
    discover_date_directories,
    load_inference_metadata,
)
from Source.training import get_device

In [ ]:
SOURCE_ROOT = PROJECT_ROOT / "Data" / "Raw" / "beach_all"
MODEL_RESULTS_DIR = PROJECT_ROOT / "Results" / "bs_convnext_tiny"
OUTPUT_ROOT = PROJECT_ROOT / "Results" / "bs_all_cells_classification"

CHECKPOINT_PATH = MODEL_RESULTS_DIR / "convnext_tiny_best.pt"
INFERENCE_METADATA_PATH = MODEL_RESULTS_DIR / "inference_metadata.json"

BATCH_SIZE = 128
NUM_WORKERS = 0
BASE_SUFFIX = "r"

print(f"Project root:       {PROJECT_ROOT}", flush=True)
print(f"Source root:        {SOURCE_ROOT}", flush=True)
print(f"Model checkpoint:   {CHECKPOINT_PATH}", flush=True)
print(f"Inference metadata: {INFERENCE_METADATA_PATH}", flush=True)
print(f"Output root:        {OUTPUT_ROOT}", flush=True)

In [ ]:
print("Checking required input paths...", flush=True)
if not SOURCE_ROOT.exists():
    raise FileNotFoundError(f"External source directory not found: {SOURCE_ROOT}")

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"ConvNeXt checkpoint not found: {CHECKPOINT_PATH}")

if not INFERENCE_METADATA_PATH.exists():
    raise FileNotFoundError(
        f"Inference metadata not found: {INFERENCE_METADATA_PATH}. "
        "Rerun the ConvNeXt training notebook so it saves inference_metadata.json."
    )

print("Loading inference metadata...", flush=True)
metadata = load_inference_metadata(INFERENCE_METADATA_PATH)
idx_to_class_code = {
    int(index): str(class_code)
    for index, class_code in metadata["idx_to_class_code"].items()
}
if "context_radius" in metadata:
    context_radius = int(metadata["context_radius"])
elif "context_radius" in metadata.get("best_config", {}):
    context_radius = int(metadata["best_config"]["context_radius"])
else:
    legacy_context_size = int(metadata.get("context_size", 0))
    context_radius = 0 if legacy_context_size <= 0 else legacy_context_size // 2

print(f"Image size:       {metadata['image_size']}", flush=True)
print(f"Context radius:   {context_radius}", flush=True)
print(f"Context window:   {2 * context_radius + 1}x{2 * context_radius + 1}", flush=True)
print(f"Feature suffixes: {metadata['feature_suffixes']}", flush=True)
print(f"Class mapping:    {idx_to_class_code}", flush=True)

In [ ]:
print("Selecting compute device...", flush=True)
device = get_device()
print(f"Device: {device}", flush=True)
if device.type == "cuda":
    print(f"CUDA devices available: {torch.cuda.device_count()}", flush=True)
    for index in range(torch.cuda.device_count()):
        print(f"CUDA device {index}: {torch.cuda.get_device_name(index)}", flush=True)

In [ ]:
print("Discovering date folders...", flush=True)
date_dirs = discover_date_directories(SOURCE_ROOT)
print(f"Date folders found: {len(date_dirs)}", flush=True)
print("This notebook uses a single-pass workflow: each date folder is scanned once using BASE_SUFFIX, then classified immediately.", flush=True)

In [ ]:
print("Starting all-cells classification...", flush=True)
print("All-cells classification finished.", flush=True)
written_summary = classify_date_directories(
    date_dirs=date_dirs,
    source_root=SOURCE_ROOT,
    output_root=OUTPUT_ROOT,
    checkpoint_path=CHECKPOINT_PATH,
    metadata=metadata,
    idx_to_class_code=idx_to_class_code,
    context_radius=context_radius,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    base_suffix=BASE_SUFFIX,
    device=device,
)
print("All-cells classification finished.", flush=True)
written_summary